# 🚕 Machine Learning Mini-Pipeline

## Let's _tax_-see how we get on

In this notebook, we'll build a machine learning mini-pipeline focussing on our NY taxi data to train a simple model and gain insights. 

### Outline of Notebook:
1. **📊 Data Loading & Exploration** - How to load and understand your data
2. **🔧 Data Preparation** - Getting data ready for machine learning
3. **🤖 Model Training** - Teaching a computer to make predictions
4. **📈 Model Evaluation** - Checking how good your predictions are
5. **💾 Model Tracking with MLflow** - Saving and organizing your experiments
6. **🚀 Making Predictions** - Using your model on new data

This should be _fare_-ly good 🚕

   
## 📚 Part 1: Setting Up Our Tools

First, we need to install and import the libraries we'll use. Think of libraries as toolboxes - each one contains specific tools for different tasks.

### Key Libraries We'll Use:
- **pyspark**: For distributed data processing across the full dataset (handles millions of rows)
- **pandas**: For small aggregated results when plotting (not for full dataset)
- **numpy**: For mathematical operations
- **sklearn**: For machine learning algorithms
- **mlflow**: For tracking our experiments
- **matplotlib/seaborn**: For creating visualisations

In [0]:
# Install required packages (run this only once)
!pip install pandas numpy scikit-learn mlflow matplotlib seaborn -q

print("✅ Packages installed successfully!")

In [0]:
# Import the libraries we need
import pandas as pd  # pd is a nickname for pandas
import numpy as np   # np is a nickname for numpy
import matplotlib.pyplot as plt  # for plotting graphs
import seaborn as sns  # for prettier graphs
import mlflow  # for tracking experiments
import warnings
warnings.filterwarnings('ignore')  # Hide unnecessary warnings

# Set up nice graph styles
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✅ All libraries imported successfully!")
print(f"You're using MLflow version: {mlflow.__version__}")

### 🎯 Python Check

Let's make sure everything is working!

In [0]:
# EXERCISE 1: Complete this code
# TODO: Create a variable called 'my_name' with your name
my_name = "Team 3"  # Replace ___ with your name in quotes

# TODO: Create a variable called 'favorite_number' with any number
favorite_number = 37  # Replace ___ with any number (no quotes)

# TODO: Print your name and favorite number
# Hint: Use print() and f-strings like f"Hello {my_name}!"
print(f"Hello {my_name}, the number is {favorite_number}")

   
## 📊 Part 2: Loading and Understanding Your Data

We'll work with our team's gold layer taxi trip dataset from Unity Catalog. Imagine you're helping a taxi company predict trip fares based on trip characteristics.

### The Taxi Dataset Contains:
- **Trip details**: `trip_distance`, `trip_duration_minutes`, `passenger_count`, `store_and_fwd_flag`
- **Fare breakdown**: `fare_amount`, `extra`, `mta_tax`, `tip_amount`, `tolls_amount`, `improvement_surcharge`, `total_amount`
- **Derived metrics**: `revenue_per_mile`, `tip_pct`, `fare_per_minute`, `fare_per_passenger`, `total_charges`, `effective_rate`, `avg_speed_mph`
- **Dimension keys**: `vendor_key`, `rate_code_key`, `payment_type_key`, `pickup_location_key`, `dropoff_location_key`, plus datetime keys
- **Target to predict**: `total_amount` 🚕

In [0]:
# Load the FULL taxi trip dataset as a Spark DataFrame (distributed - no memory issues)
taxi_data = spark.table("students_data.`team3-taxi`.fact_trip")
total_rows = taxi_data.count()

print("✅ Data loaded successfully!")
print(f"🚕 Full dataset: {total_rows:,} trips with {len(taxi_data.columns)} features")
print(f"💡 Data stays distributed across the cluster - no OutOfMemoryError!")

In [0]:
# Let's look at the first 5 rows of our data
print("First 5 trips in our dataset:")
print("=" * 50)
display(taxi_data.limit(5))  # .limit(5) shows the first 5 rows

In [0]:
# Understanding our data better
from pyspark.sql import functions as F

print("📊 Dataset Information:")
print("=" * 50)
print(f"Total number of trips: {total_rows:,}")
print(f"Number of features: {len(taxi_data.columns)}")
print(f"\nColumn names and their data types:")
for field in taxi_data.schema.fields:
    print(f"  {field.name}: {field.dataType.simpleString()}")

print("\n" + "=" * 50)
print("Basic statistics of our data:")
display(taxi_data.describe())  # .describe() shows statistics for all numeric columns

### 🎯 Exploring the Data

Now it's time to explore the data!

In [0]:
# EXERCISE 2.1: Find the average fare amount
average_fare = taxi_data.select(F.mean('fare_amount')).first()[0]
print(f"Average fare amount: ${average_fare:.2f}")

# EXERCISE 2.2: Find the highest total amount in the dataset
highest_total = taxi_data.select(F.max('total_amount')).first()[0]
print(f"Highest total amount: ${highest_total:.2f}")

# EXERCISE 2.3: Count how many trips have total amount of $50 or higher
high_fare_trips = taxi_data.filter(F.col('total_amount') >= 50)
high_fare_percentage = high_fare_trips.count() / total_rows * 100
print(f"Number of high fare trips ($50+): {high_fare_trips.count():,} ({high_fare_percentage:.2f}% of total)")

## 📈 Part 3: Visualising The Data

A picture is worth a thousand words! Let's create some graphs to better understand our taxi data.

In [0]:
# Example: Create a histogram of fare amount distribution using the FULL dataset
# Spark aggregates all 46M rows; we only collect the small binned result for plotting
bin_width = 3.33
fare_hist = (taxi_data
    .filter(F.col('fare_amount').between(0, 100))
    .withColumn('fare_bin', F.floor(F.col('fare_amount') / bin_width) * bin_width)
    .groupBy('fare_bin')
    .agg(F.count('*').alias('trip_count'))
    .orderBy('fare_bin')
    .toPandas()  # Only ~30 aggregated rows collected, not 46M!
)

plt.figure(figsize=(8, 6))
plt.bar(fare_hist['fare_bin'], fare_hist['trip_count'], width=bin_width, color='skyblue', edgecolor='black')
plt.title('Distribution of Taxi Fare Amounts (up to $100)', fontsize=14)
plt.xlabel('Fare Amount ($)')
plt.ylabel('Number of Trips')
plt.ticklabel_format(style='plain', axis='y')
plt.show()

print(f"📊 Observation: Histogram built from {total_rows:,} trips across the full dataset")
print("Most fares are concentrated in the lower range, with a long tail of expensive trips")

   
### 🎯 More Visualisations...

In [0]:
# Explore relationship between trip distance and total amount
# Spark aggregates the FULL dataset; we only collect the small summary for plotting

distance_bin_width = 0.5  # half-mile bins
scatter_agg = (taxi_data
    .filter((F.col('trip_distance').between(0, 30)) & (F.col('total_amount').between(0, 100)))
    .withColumn('distance_bin', F.round(F.col('trip_distance') / distance_bin_width) * distance_bin_width)
    .groupBy('distance_bin')
    .agg(
        F.mean('total_amount').alias('avg_total_amount'),
        F.count('*').alias('trip_count')
    )
    .orderBy('distance_bin')
    .toPandas()  # Only ~60 aggregated rows collected, not 46M!
)

plt.figure(figsize=(8, 6))
plt.scatter(scatter_agg['distance_bin'], scatter_agg['avg_total_amount'],
            s=scatter_agg['trip_count'] / scatter_agg['trip_count'].max() * 200,  # Size by volume
            alpha=0.7, color='teal', edgecolors='black', linewidth=0.5)
plt.title('Average Total Amount by Trip Distance - Full Dataset', fontsize=14)
plt.xlabel('Trip Distance (miles)')
plt.ylabel('Average Total Amount ($)')
plt.grid(True, alpha=0.3)
plt.show()

# Bonus: Calculate correlation across the full dataset
correlation = taxi_data.filter(
    (F.col('trip_distance').between(0, 30)) & (F.col('total_amount').between(0, 100))
).stat.corr('trip_distance', 'total_amount')
print(f"📊 Correlation between trip distance and total amount (full dataset): {correlation:.3f}")
print("💡 Bubble size represents the number of trips at each distance")

## 🔧 Part 4: Preparing Data for Machine Learning

Before we can train a model, we need to prepare our data:

1. **Separate features (X) from target (y)**: Features are what we use to predict, target is what we want to predict
2. **Split into training and testing sets**: Train on some data, test on different data
3. **Scale the features**: Make all features have similar ranges

### Why Split Data?
Imagine studying for an exam. If you only practice the exact questions that will be on the test, you're memorizing, not learning! Similarly, we train our model on one set of data and test it on different data to see if it truly learned patterns.

In [0]:
# Step 1: Separate features (X) and target (y)
# Features = everything we use to make predictions
# Target = what we want to predict (total_amount)

# Drop non-predictive columns and the target
columns_to_drop = ['total_amount', 'trip_id', 'store_and_fwd_flag', 'total_charges', 'effective_rate']
feature_columns = [c for c in taxi_data.columns if c not in columns_to_drop]

X = taxi_data.select(feature_columns)  # Features
y = taxi_data.select('total_amount')   # Target

print("✅ Features and target separated!")
print(f"Features: {X.count():,} rows x {len(feature_columns)} columns")
print(f"Target: {y.count():,} rows")
print(f"\nFeatures we'll use for prediction:")
print(feature_columns)
print(f"\n🚫 Dropped columns: {columns_to_drop}")
print("   (trip_id is a unique ID, store_and_fwd_flag is a string,")
print("    total_charges & effective_rate leak the target)")


In [0]:
# Step 2: Split into training and testing sets
# Spark's randomSplit handles this natively on the full distributed dataset

# Keep features and target together, drop nulls for clean ML input
ml_data = taxi_data.select(feature_columns + ['total_amount']).dropna()

# Split: 80% for training, 20% for testing
train_data, test_data = ml_data.randomSplit([0.8, 0.2], seed=42)

train_count = train_data.count()
test_count = test_data.count()
total_ml_rows = train_count + test_count

print("✅ Data split complete!")
print(f"Training set: {train_count:,} trips")
print(f"Testing set: {test_count:,} trips")
print(f"\nThis means we'll train on {train_count/total_ml_rows*100:.1f}% of the data")
print(f"And test on {test_count/total_ml_rows*100:.1f}% of the data")

In [0]:
# Step 3: Assemble features into a vector and scale them
# Spark ML requires all features combined into a single vector column
from pyspark.ml.feature import VectorAssembler, StandardScaler as SparkScaler

# Combine all feature columns into one vector column
assembler = VectorAssembler(inputCols=feature_columns, outputCol='features_raw', handleInvalid='skip')

# Create a scaler (same idea as sklearn's StandardScaler)
scaler = SparkScaler(inputCol='features_raw', outputCol='features', withMean=True, withStd=True)

# Fit on training data, then transform both sets
train_assembled = assembler.transform(train_data)
scaler_model = scaler.fit(train_assembled)
train_scaled = scaler_model.transform(train_assembled)

test_assembled = assembler.transform(test_data)
test_scaled = scaler_model.transform(test_assembled)

print("✅ Features assembled and scaled!")
print(f"\n{len(feature_columns)} feature columns combined into one 'features' vector column")
print(f"Training set: {train_count:,} rows")
print(f"Testing set: {test_count:,} rows")
print("\nWhy scaling matters:")
print("  Before scaling - trip_distance and fare_amount have very different ranges")
print("  After scaling - All features have mean≈0 and standard deviation≈1")

### 🎯 Exercise 4: Understanding Data Splits

Looking a bit more at data splitting.

In [0]:
# EXERCISE 4: Understanding data splits

# 4.1: Calculate the number of trips for different split ratios
total_clean_trips = total_ml_rows

# If we use 70% for training, how many trips would that be?
train_70_percent = int(total_clean_trips * 0.7)
test_30_percent = total_clean_trips - train_70_percent

print(f"With 70/30 split:")
print(f"  Training: {train_70_percent:,} trips")
print(f"  Testing: {test_30_percent:,} trips")

# 4.2: Check the distribution of total_amount in train and test sets
train_stats = train_data.select(
    F.mean('total_amount').alias('mean'),
    F.stddev('total_amount').alias('std'),
    F.min('total_amount').alias('min'),
    F.max('total_amount').alias('max')
).first()

test_stats = test_data.select(
    F.mean('total_amount').alias('mean'),
    F.stddev('total_amount').alias('std'),
    F.min('total_amount').alias('min'),
    F.max('total_amount').alias('max')
).first()

print(f"\nTotal amount distribution in training set:")
print(f"  Mean: ${train_stats['mean']:.2f}, Std: ${train_stats['std']:.2f}")
print(f"  Range: ${train_stats['min']:.2f} to ${train_stats['max']:.2f}")

print(f"\nTotal amount distribution in test set:")
print(f"  Mean: ${test_stats['mean']:.2f}, Std: ${test_stats['std']:.2f}")
print(f"  Range: ${test_stats['min']:.2f} to ${test_stats['max']:.2f}")

# Question: Why is it important that both sets have similar distributions?
# Your answer: So the model is tested on data that represents the same patterns it learned from

   
## 🤖 Part 5: Training Your First Machine Learning Model

Now for the exciting part - training a model to predict taxi fare totals!

### What is a Machine Learning Model?
Think of a model as a student learning from examples:
1. **Training**: Show the student many examples (trips with known fare amounts)
2. **Learning**: The student finds patterns (e.g., "longer distance usually means higher fare")
3. **Predicting**: The student can now estimate fares for new trips

We'll use a **Random Forest** model - imagine having 100 fare estimation experts vote on the total amount!

In [0]:
# Import the Random Forest model from Spark MLlib
from pyspark.ml.regression import RandomForestRegressor as SparkRF
from pyspark.ml.evaluation import RegressionEvaluator

# Create a Random Forest model
# numTrees = number of "expert trees" in our forest
# seed = ensures we get the same results each time
model = SparkRF(
    featuresCol='features',
    labelCol='total_amount',
    numTrees=100,
    seed=42
)

print("🌲 Random Forest model created!")
print(f"This model will use {model.getNumTrees()} decision trees to make predictions")

In [0]:
# Train the model on the full training set
print("🎓 Training the model on the full distributed dataset...")
trained_model = model.fit(train_scaled)
print("✅ Model training complete!")

# Make predictions on test set
predictions_df = trained_model.transform(test_scaled)

print(f"\n📊 Made predictions for {test_count:,} test trips")
print(f"\nFirst 5 predictions vs actual values:")
sample_preds = predictions_df.select('total_amount', 'prediction').limit(5).collect()
for i, row in enumerate(sample_preds):
    print(f"Trip {i+1}: Predicted=${row['prediction']:.2f}, Actual=${row['total_amount']:.2f}")

In [0]:
# Evaluate the model performance using Spark's RegressionEvaluator
evaluator_mae = RegressionEvaluator(labelCol='total_amount', predictionCol='prediction', metricName='mae')
evaluator_rmse = RegressionEvaluator(labelCol='total_amount', predictionCol='prediction', metricName='rmse')
evaluator_r2 = RegressionEvaluator(labelCol='total_amount', predictionCol='prediction', metricName='r2')

mae = evaluator_mae.evaluate(predictions_df)
rmse = evaluator_rmse.evaluate(predictions_df)
r2 = evaluator_r2.evaluate(predictions_df)

print("📊 Model Performance Metrics:")
print("=" * 50)
print(f"Mean Absolute Error (MAE): ${mae:.3f}")
print(f"  → On average, predictions are off by ${mae:.2f}")
print(f"\nRoot Mean Squared Error (RMSE): ${rmse:.3f}")
print(f"  → Penalizes large errors more than MAE")
print(f"\nR² Score: {r2:.3f}")
print(f"  → Model explains {r2*100:.1f}% of fare variation")
print("\n" + "=" * 50)

if mae < 2:
    print("🎉 Excellent! Predictions are very close to actual fares!")
elif mae < 5:
    print("👍 Good job! Predictions are reasonably accurate!")
else:
    print("🤔 Room for improvement, but it's a good start!")

In [0]:
# Visualize predictions vs actual values
# Aggregate in Spark, collect small result for plotting
pred_vs_actual = (predictions_df
    .select(
        F.round(F.col('total_amount'), 0).alias('actual_bin'),
        F.round(F.col('prediction'), 0).alias('predicted_bin')
    )
    .filter((F.col('actual_bin').between(0, 100)) & (F.col('predicted_bin').between(0, 100)))
    .groupBy('actual_bin')
    .agg(
        F.mean('predicted_bin').alias('avg_predicted'),
        F.count('*').alias('trip_count')
    )
    .orderBy('actual_bin')
    .toPandas()
)

plt.figure(figsize=(10, 6))
plt.scatter(pred_vs_actual['actual_bin'], pred_vs_actual['avg_predicted'],
            s=pred_vs_actual['trip_count'] / pred_vs_actual['trip_count'].max() * 200,
            alpha=0.7, color='darkblue', edgecolors='black', linewidth=0.5)

# Perfect prediction line
min_val = 0
max_val = 100
plt.plot([min_val, max_val], [min_val, max_val], 'r--', label='Perfect Prediction')

plt.xlabel('Actual Total Amount ($)', fontsize=12)
plt.ylabel('Avg Predicted Total Amount ($)', fontsize=12)
plt.title('Model Predictions vs Actual Taxi Fares - Full Dataset', fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("💡 Points closer to the red line = better predictions!")
print("💡 Bubble size represents the number of trips at each fare level")

   
### 🎯 Exercise 5: Feature Importance

Which features are most important for predicting taxi fare totals?

In [0]:
# EXERCISE 5: Analyze feature importance
# Spark's Random Forest model provides feature importances directly

# Get feature importances from the trained model
importances = trained_model.featureImportances.toArray()

# Create a pandas DataFrame for easy visualization (only ~25 rows)
importance_df = pd.DataFrame({
    'feature': feature_columns,
    'importance': importances
})

# Sort by importance
importance_df = importance_df.sort_values('importance', ascending=False)

# Plot the top 5 most important features
plt.figure(figsize=(10, 6))
top_5 = importance_df.head(5)
plt.barh(top_5['feature'], top_5['importance'], color='salmon')
plt.xlabel('Importance')
plt.title('Top 5 Most Important Features for Taxi Fare Prediction')
plt.gca().invert_yaxis()
plt.show()

# Print the most important feature
most_important = importance_df.iloc[0]
print(f"\n🏆 Most important feature: {most_important['feature']}")
print(f"   Importance score: {most_important['importance']:.3f}")

   
## 💾 Part 6: Tracking Experiments with MLflow

### What is MLflow?
MLflow is like a lab notebook for data scientists. It helps you:
- **Track experiments**: Record what you tried and what happened
- **Compare models**: See which approach worked best
- **Save models**: Store your trained models for later use
- **Share work**: Let others use your models

Think of it as Instagram for your machine learning models - you can save, organize, and share them!

In [0]:
# Start MLflow tracking
import mlflow
import mlflow.spark

# Set experiment name (like creating a folder for this project)
mlflow.set_experiment("/Users/neil.obriain@kainos.com/taxi-fare-prediction")

print("🚀 Starting MLflow experiment tracking...")
print("This will record:")
print("  ✓ Model parameters")
print("  ✓ Performance metrics")
print("  ✓ The trained model itself")
print("  ✓ Timestamps and duration")

In [0]:
# Train and track a model with MLflow
with mlflow.start_run(run_name="random_forest_baseline"):
    # Log model parameters
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("test_size", 0.2)
    mlflow.log_param("seed", 42)
    mlflow.log_param("num_features", len(feature_columns))
    
    # Train Spark MLlib Random Forest model
    rf = SparkRF(featuresCol='features', labelCol='total_amount', numTrees=100, seed=42)
    rf_trained = rf.fit(train_scaled)
    
    # Make predictions
    rf_predictions = rf_trained.transform(test_scaled)
    
    # Calculate metrics using Spark evaluators
    rf_mae = evaluator_mae.evaluate(rf_predictions)
    rf_rmse = evaluator_rmse.evaluate(rf_predictions)
    rf_r2 = evaluator_r2.evaluate(rf_predictions)
    
    # Log metrics
    mlflow.log_metric("mae", rf_mae)
    mlflow.log_metric("rmse", rf_rmse)
    mlflow.log_metric("r2", rf_r2)
    
    # Note: mlflow.spark.log_model requires a UC Volume path on shared clusters.
    # To log the model, set MLFLOW_DFS_TMP to a /Volumes/... path first.
    
    print("✅ Experiment tracked successfully!")
    print(f"\n📊 Results:")
    print(f"  MAE: ${rf_mae:.3f}")
    print(f"  RMSE: ${rf_rmse:.3f}")
    print(f"  R²: {rf_r2:.3f}")
    
    # Get run ID for later reference
    run_id = mlflow.active_run().info.run_id
    print(f"\n🔖 Run ID: {run_id}")

   
### 🎯 Exercise 6: Experiment with Different Models

Try training a different model and compare results!

In [0]:
# EXERCISE 6: Train a simpler model and track it with MLflow
from pyspark.ml.regression import LinearRegression as SparkLR

# TODO: Complete the MLflow tracking code
with mlflow.start_run(run_name="LR-run"):
    # Log that we're using Linear Regression
    mlflow.log_param("model_type", "linear_regression")
    
    # Create and train a Spark MLlib Linear Regression model
    lr = SparkLR(featuresCol='features', labelCol='total_amount', maxIter=100)
    lr_trained = lr.fit(train_scaled)
    
    # Make predictions
    lr_predictions = lr_trained.transform(test_scaled)
    
    # Calculate MAE
    lr_mae = evaluator_mae.evaluate(lr_predictions)
    lr_rmse = evaluator_rmse.evaluate(lr_predictions)
    lr_r2 = evaluator_r2.evaluate(lr_predictions)
    
    # Log metrics
    mlflow.log_metric("mae", lr_mae)
    mlflow.log_metric("rmse", lr_rmse)
    mlflow.log_metric("r2", lr_r2)
    
    print(f"Linear Regression MAE: ${lr_mae:.3f}")
    
# Compare with Random Forest
print(f"\n📊 Model Comparison:")
print(f"Random Forest MAE: ${rf_mae:.3f}")
print(f"Linear Regression MAE: ${lr_mae:.3f}")

if rf_mae < lr_mae:
    print("\n🏆 Random Forest performs better!")
else:
    print("\n🏆 Linear Regression performs better!")

   
## 🚀 Part 7: Making Predictions on New Trips

Now let's use our trained model to predict fares for completely new taxi trips!

In [0]:
# Create a function to predict taxi fare using the Spark pipeline
def predict_taxi_fare(trip_features, trained_model, assembler, scaler_model):
    """
    Predict the total fare for a taxi trip given its features.
    
    Args:
        trip_features: Dictionary with trip characteristics
        trained_model: Trained Spark ML model
        assembler: Fitted VectorAssembler
        scaler_model: Fitted StandardScaler model
    
    Returns:
        Predicted total amount
    """
    # Create a Spark DataFrame from the trip features
    trip_df = spark.createDataFrame([trip_features])
    
    # Assemble and scale features
    trip_assembled = assembler.transform(trip_df)
    trip_scaled = scaler_model.transform(trip_assembled)
    
    # Make prediction
    prediction = trained_model.transform(trip_scaled)
    return prediction.select('prediction').first()[0]

# Example: Predict fare for a new trip
new_trip = {col: 0.0 for col in feature_columns}  # Start with defaults
new_trip.update({
    'vendor_key': 1,
    'rate_code_key': 1,
    'payment_type_key': 1,
    'pickup_location_key': 7355375,
    'dropoff_location_key': 8083728,
    'pickup_datetime_day_key': 15,
    'dropoff_datetime_day_key': 15,
    'pickup_datetime_hour_key': 500,
    'dropoff_datetime_hour_key': 501,
    'pickup_datetime_minute_key': 30000,
    'dropoff_datetime_minute_key': 30015,
    'trip_distance': 3.5,
    'trip_duration_minutes': 15,
    'fare_amount': 14.0,
    'extra': 0.5,
    'mta_tax': 0.5,
    'tip_amount': 2.0,
    'tolls_amount': 0.0,
    'improvement_surcharge': 0.3,
    'passenger_count': 1,
    'revenue_per_mile': 4.0,
    'tip_pct': 0.14,
    'fare_per_minute': 0.93,
    'fare_per_passenger': 14.0,
    'avg_speed_mph': 14.0
})

predicted_fare = predict_taxi_fare(new_trip, trained_model, assembler, scaler_model)

print("🚕 New Trip Analysis:")
print("=" * 50)
print("Trip Details:")
print(f"  Distance: {new_trip['trip_distance']} miles")
print(f"  Duration: {new_trip['trip_duration_minutes']} minutes")
print(f"  Passengers: {new_trip['passenger_count']}")
print(f"  Base fare: ${new_trip['fare_amount']:.2f}")
print("\n" + "=" * 50)
print(f"📊 Predicted Total Amount: ${predicted_fare:.2f}")

   
### 🎯 Exercise 7: Predict Your Own Trip

Create your own taxi trip by adjusting the trip characteristics!

In [0]:
# EXERCISE 7: Design your own taxi trip and predict its fare
# TODO: Modify the values to create your custom trip

my_custom_trip = {col: 0.0 for col in feature_columns}
my_custom_trip.update({
    'vendor_key': 2,
    'rate_code_key': 1,
    'payment_type_key': 1,
    'pickup_location_key': 7355375,
    'dropoff_location_key': 21382754,
    'pickup_datetime_day_key': 20,
    'dropoff_datetime_day_key': 20,
    'pickup_datetime_hour_key': 400,
    'dropoff_datetime_hour_key': 401,
    'pickup_datetime_minute_key': 24000,
    'dropoff_datetime_minute_key': 24045,
    'trip_distance': 10.0,         # Try: 0.5 to 30 miles
    'trip_duration_minutes': 35,    # Try: 2 to 120 minutes
    'fare_amount': 35.0,           # Try: 2.5 to 100
    'extra': 1.0,                  # Try: 0 to 5
    'mta_tax': 0.5,
    'tip_amount': 7.0,             # Try: 0 to 50
    'tolls_amount': 5.0,           # Try: 0 to 30
    'improvement_surcharge': 0.3,
    'passenger_count': 3,           # Try: 1 to 6
    'revenue_per_mile': 3.5,
    'tip_pct': 0.2,
    'fare_per_minute': 1.0,
    'fare_per_passenger': 11.67,
    'avg_speed_mph': 17.1
})

# Predict fare
my_trip_fare = predict_taxi_fare(my_custom_trip, trained_model, assembler, scaler_model)

print("🚕 Your Custom Trip:")
print(f"  Distance: {my_custom_trip['trip_distance']} miles")
print(f"  Duration: {my_custom_trip['trip_duration_minutes']} minutes")
print(f"  Passengers: {my_custom_trip['passenger_count']}")
print(f"Predicted Total: ${my_trip_fare:.2f}")

# Challenge: Can you create a trip that totals over $100?

## 🎉 Part 8: Final Project - Build Your Complete Pipeline

Now it's time to put everything together! Create a complete machine learning pipeline from start to finish.

In [0]:
# FINAL PROJECT: Complete ML Pipeline
import os
os.environ["MLFLOW_DFS_TMP"] = "/Volumes/students_data/team3-taxi/mlflow_tmp"

def complete_taxi_fare_pipeline(table_name, model_type='random_forest'):
    """
    Complete Spark MLlib pipeline for taxi fare prediction.
    Uses only user-friendly features that a passenger would know:
      - trip_distance, trip_duration_minutes, passenger_count,
        vendor_key, rate_code_key, payment_type_key
      - hour (0-23), day_of_week (1-7), is_weekend (from dim tables)
    
    Steps:
    1. Load data and join dimension tables for time features
    2. Prepare user-friendly features and target
    3. Split data
    4. Assemble and scale features
    5. Train model
    6. Evaluate model
    7. Track with MLflow and save model
    """
    import mlflow
    import mlflow.spark
    from mlflow.models import infer_signature
    from pyspark.ml.feature import VectorAssembler, StandardScaler as SparkScaler
    from pyspark.ml.regression import RandomForestRegressor as SparkRF, LinearRegression as SparkLR
    from pyspark.ml.evaluation import RegressionEvaluator
    from pyspark.sql.functions import col
    
    print("\U0001f680 Starting Taxi Fare Prediction Pipeline...")
    print("=" * 50)
    
    # Step 1: Load data and join dimension tables for time-of-day features
    print("\U0001f4ca Loading data and joining time dimensions...")
    trips = spark.table(table_name)
    dim_hour = spark.table("students_data.`team3-taxi`.dim_datetime_hour") \
        .select(
            col("datetime_hour_key"),
            col("hour").alias("pickup_hour"),
            col("day_of_week").alias("pickup_day_of_week"),
            col("is_weekend").alias("pickup_is_weekend")
        )
    
    data = trips.join(
        dim_hour,
        trips.pickup_datetime_hour_key == dim_hour.datetime_hour_key,
        "left"
    ).withColumn("pickup_is_weekend", col("pickup_is_weekend").cast("int"))
    
    total = data.count()
    print(f"\u2713 Loaded {total:,} trips with time features")
    
    # Step 2: User-friendly features only
    # Things a passenger would know before/during a trip
    feat_cols = [
        'trip_distance',           # How far the trip is (miles)
        'trip_duration_minutes',   # How long the trip takes
        'passenger_count',         # Number of passengers
        'vendor_key',              # Taxi vendor / company (1 or 2)
        'rate_code_key',           # Rate type (1=Standard, 2=JFK, etc.)
        'payment_type_key',        # Payment method (1=Card, 2=Cash, etc.)
        'pickup_hour',             # Hour of day (0-23)
        'pickup_day_of_week',      # Day of week (1=Sun .. 7=Sat)
        'pickup_is_weekend',       # Weekend flag (0 or 1)
    ]
    print(f"\u2713 Using {len(feat_cols)} user-friendly features: {feat_cols}")
    
    ml_df = data.select(feat_cols + ['total_amount']).dropna()
    
    # Step 3: Split data
    train_df, test_df = ml_df.randomSplit([0.8, 0.2], seed=42)
    print(f"\u2713 Split data: {train_df.count():,} train, {test_df.count():,} test")
    
    # Step 4: Assemble and scale features
    asm = VectorAssembler(inputCols=feat_cols, outputCol='features_raw', handleInvalid='skip')
    scl = SparkScaler(inputCol='features_raw', outputCol='features', withMean=True, withStd=True)
    
    train_asm = asm.transform(train_df)
    scl_model = scl.fit(train_asm)
    train_ready = scl_model.transform(train_asm)
    test_ready = scl_model.transform(asm.transform(test_df))
    print("\u2713 Features assembled and scaled")
    
    # Step 5: Train model
    print(f"\U0001f916 Training {model_type} model...")
    if model_type == 'random_forest':
        ml_model = SparkRF(featuresCol='features', labelCol='total_amount', numTrees=100, seed=42)
    else:
        ml_model = SparkLR(featuresCol='features', labelCol='total_amount', maxIter=100)
    
    fitted_model = ml_model.fit(train_ready)
    print("\u2713 Model trained")
    
    # Step 6: Evaluate
    preds = fitted_model.transform(test_ready)
    eval_mae = RegressionEvaluator(labelCol='total_amount', predictionCol='prediction', metricName='mae')
    eval_r2 = RegressionEvaluator(labelCol='total_amount', predictionCol='prediction', metricName='r2')
    pipeline_mae = eval_mae.evaluate(preds)
    pipeline_r2 = eval_r2.evaluate(preds)
    
    print(f"\n\U0001f4ca Results:")
    print(f"Model: {model_type}")
    print(f"MAE: ${pipeline_mae:.3f}")
    print(f"R\u00b2:  {pipeline_r2:.3f}")
    
    # Step 7: Track with MLflow and save model
    print("\n\U0001f4be Saving model to MLflow...")
    mlflow.set_experiment("/Users/neil.obriain@kainos.com/taxi-fare-prediction")
    
    with mlflow.start_run(run_name=f"{model_type}_user_friendly"):
        # Log parameters
        mlflow.log_param("model_type", model_type)
        mlflow.log_param("features", feat_cols)
        mlflow.log_param("num_features", len(feat_cols))
        mlflow.log_param("train_size", train_df.count())
        mlflow.log_param("test_size", test_df.count())
        
        # Log metrics
        mlflow.log_metric("mae", pipeline_mae)
        mlflow.log_metric("r2", pipeline_r2)
        
        # Infer signature from predictions
        sample_input = preds.select(feat_cols).limit(5).toPandas()
        sample_output = preds.select("prediction").limit(5).toPandas()
        signature = infer_signature(sample_input, sample_output)
        
        # Log and save the trained Spark model
        mlflow.spark.log_model(
            fitted_model,
            artifact_path="taxi_fare_model",
            signature=signature,
            input_example=sample_input.head(1),
        )
        
        run_id = mlflow.active_run().info.run_id
        print(f"\u2713 Model saved to MLflow (run_id: {run_id})")
        print(f"  Load later with: mlflow.spark.load_model('runs:/{run_id}/taxi_fare_model')")
    
    return fitted_model, asm, scl_model, feat_cols, pipeline_mae

# Run the pipeline
final_model, final_assembler, final_scaler, final_features, final_mae = complete_taxi_fare_pipeline(
    "students_data.`team3-taxi`.fact_trip"
)

print("\n\U0001f389 Pipeline completed successfully!")

In [0]:
# Simple fare predictor - just provide what a passenger would know!

def predict_fare(trip_distance, trip_duration_minutes, passenger_count=1,
                 pickup_hour=12, pickup_day_of_week=3, pickup_is_weekend=0,
                 vendor_key=1, rate_code_key=1, payment_type_key=1):
    """
    Predict taxi fare from simple, user-friendly inputs.

    Args:
        trip_distance:          Distance in miles (e.g. 3.5)
        trip_duration_minutes:  Estimated trip time in minutes (e.g. 15)
        passenger_count:        Number of passengers (1-6, default 1)
        pickup_hour:            Hour of pickup (0-23, default 12 noon)
        pickup_day_of_week:     Day of week (1=Sun, 2=Mon, ... 7=Sat, default 3=Tue)
        pickup_is_weekend:      Weekend flag (0=weekday, 1=weekend, default 0)
        vendor_key:             Taxi vendor (1 or 2, default 1)
        rate_code_key:          Rate code (1=Standard, 2=JFK, 3=Newark,
                                4=Nassau/Westchester, 5=Negotiated, default 1)
        payment_type_key:       Payment type (1=Credit card, 2=Cash,
                                3=No charge, 4=Dispute, default 1)
    Returns:
        Predicted total fare amount ($)
    """
    trip = {
        'trip_distance':         float(trip_distance),
        'trip_duration_minutes': float(trip_duration_minutes),
        'passenger_count':       float(passenger_count),
        'vendor_key':            float(vendor_key),
        'rate_code_key':         float(rate_code_key),
        'payment_type_key':      float(payment_type_key),
        'pickup_hour':           float(pickup_hour),
        'pickup_day_of_week':    float(pickup_day_of_week),
        'pickup_is_weekend':     float(pickup_is_weekend),
    }
    trip_df = spark.createDataFrame([trip])
    trip_assembled = final_assembler.transform(trip_df)
    trip_scaled = final_scaler.transform(trip_assembled)
    prediction = final_model.transform(trip_scaled)
    return round(prediction.select('prediction').first()[0], 2)


# ── Example predictions ──────────────────────────────────────────
print("\U0001f695 Taxi Fare Predictions")
print("=" * 55)

examples = [
    {"desc": "Short weekday morning ride (1 mi, 5 min, 8am Tue)",
     "args": dict(trip_distance=1.0, trip_duration_minutes=5,
                  passenger_count=1, pickup_hour=8, pickup_day_of_week=3)},
    {"desc": "Medium weekday afternoon ride (5 mi, 20 min, 2pm Wed)",
     "args": dict(trip_distance=5.0, trip_duration_minutes=20,
                  passenger_count=2, pickup_hour=14, pickup_day_of_week=4)},
    {"desc": "Long weekend evening ride (15 mi, 45 min, 9pm Sat)",
     "args": dict(trip_distance=15.0, trip_duration_minutes=45,
                  passenger_count=3, pickup_hour=21, pickup_day_of_week=7, pickup_is_weekend=1)},
    {"desc": "JFK flat-rate weekday (18 mi, 50 min, 6am Mon)",
     "args": dict(trip_distance=18.0, trip_duration_minutes=50,
                  passenger_count=1, pickup_hour=6, pickup_day_of_week=2, rate_code_key=2)},
    {"desc": "Late-night weekend short hop (2 mi, 10 min, 1am Sun)",
     "args": dict(trip_distance=2.0, trip_duration_minutes=10,
                  passenger_count=2, pickup_hour=1, pickup_day_of_week=1, pickup_is_weekend=1)},
]

for ex in examples:
    fare = predict_fare(**ex["args"])
    print(f"\n  {ex['desc']}")
    print(f"  \u2192 Predicted fare: ${fare:.2f}")

print("\n" + "=" * 55)
print("\U0001f4a1 Try your own:")
print("   predict_fare(trip_distance=3.5, trip_duration_minutes=15, pickup_hour=17)")

In [0]:
# Test the model against real trips from the dataset
from pyspark.sql.functions import col, abs as spark_abs, avg, percentile_approx, try_divide, rand

# --- Build the same enriched dataset used for training ---
trips = spark.table("students_data.`team3-taxi`.fact_trip")
dim_hour = spark.table("students_data.`team3-taxi`.dim_datetime_hour") \
    .select(
        col("datetime_hour_key"),
        col("hour").alias("pickup_hour"),
        col("day_of_week").alias("pickup_day_of_week"),
        col("is_weekend").alias("pickup_is_weekend")
    )

enriched = trips.join(
    dim_hour,
    trips.pickup_datetime_hour_key == dim_hour.datetime_hour_key,
    "left"
).withColumn("pickup_is_weekend", col("pickup_is_weekend").cast("int"))

# --- Filter to valid trips only (minimum 1 mile) ---
feat_cols = [
    'trip_distance', 'trip_duration_minutes', 'passenger_count',
    'vendor_key', 'rate_code_key', 'payment_type_key',
    'pickup_hour', 'pickup_day_of_week', 'pickup_is_weekend',
]
valid_trips = (
    enriched
    .select(feat_cols + ['total_amount'])
    .dropna()
    .filter(col("trip_distance") >= 1)
    .filter(col("trip_duration_minutes") > 0)
    .filter(col("total_amount") > 0)
    .filter(col("passenger_count") > 0)
)

# --- Random sample of 200 valid trips for a good spread ---
sample_trips = valid_trips.orderBy(rand(seed=42)).limit(200)

assembled = final_assembler.transform(sample_trips)
scaled    = final_scaler.transform(assembled)
preds     = final_model.transform(scaled)

results = preds.select(
    *feat_cols,
    col("total_amount").alias("actual_fare"),
    col("prediction").alias("predicted_fare"),
    spark_abs(col("prediction") - col("total_amount")).alias("abs_error"),
    try_divide((col("prediction") - col("total_amount")) * 100, col("total_amount")).alias("pct_error"),
)

# --- Summary statistics ---
stats = results.select(
    avg("abs_error").alias("mean_abs_error"),
    percentile_approx("abs_error", 0.5).alias("median_abs_error"),
    avg(spark_abs(col("pct_error"))).alias("mean_abs_pct_error"),
).first()

print("\U0001f4ca Model Accuracy on 200 Random Valid Trips (>= 1 mile)")
print("=" * 55)
print(f"  Mean Absolute Error:    ${stats['mean_abs_error']:.2f}")
print(f"  Median Absolute Error:  ${stats['median_abs_error']:.2f}")
print(f"  Mean Abs % Error:       {stats['mean_abs_pct_error']:.1f}%")
print("=" * 55)

# --- Show predictions vs actuals, sorted by distance for readability ---
print("\n\U0001f50d Sample predictions vs actual fares:")
display(
    results.select(
        "trip_distance", "trip_duration_minutes", "pickup_hour",
        "actual_fare", "predicted_fare", "abs_error", "pct_error"
    ).orderBy("trip_distance")
)

## 📝 Summary and Next Steps

### 🎉 Congratulations! You've Completed Your First Data Science Project!

#### What You've Learned:
1. **Data Loading & Exploration** - How to load and understand large datasets with Spark
2. **Data Visualization** - Creating graphs to find patterns (aggregating in Spark, plotting with matplotlib)
3. **Data Preparation** - Splitting and scaling data for ML using Spark MLlib
4. **Model Training** - Teaching computers to predict taxi fares using distributed computing
5. **Model Evaluation** - Measuring how good predictions are with MAE, RMSE, R²
6. **MLflow Tracking** - Organizing and saving experiments
7. **Making Predictions** - Using models on new trip data

### 📊 Your Model's Performance:
- Your Random Forest model can predict taxi fares across 46M+ trips
- It learned that trip distance and fare amount are key drivers of total cost
- You tracked experiments with MLflow for reproducibility

### 🚀 Next Steps to Continue Learning:

#### Easy Next Projects:
1. **Try Different Models**: Experiment with GBTRegressor or DecisionTreeRegressor in Spark MLlib
2. **Feature Engineering**: Create new features (e.g., time of day categories, weekend flag)
3. **Hyperparameter Tuning**: Use CrossValidator to find best model settings
4. **Cross-Validation**: Better evaluate your model's performance with k-fold CV

#### Intermediate Challenges:
1. **Classification**: Instead of predicting exact fare, classify as "Short", "Medium", or "Long" trip
2. **Handle Outliers**: Some fares are extremely high - how to handle this?
3. **Join Dimension Tables**: Enrich data with vendor names, location details, payment types
4. **Try Other Datasets**: Apply these skills to different problems

#### Resources for Continued Learning:
- **Databricks Documentation**: Deep dive into Spark MLlib
- **Spark MLlib Guide**: Advanced distributed ML algorithms
- **MLflow Documentation**: Advanced experiment tracking
- **Kaggle**: Join competitions and learn from others

### 💡 Key Takeaways:
1. **Spark for Scale**: Use Spark DataFrames and MLlib for datasets that don't fit in memory
2. **No Model is Perfect**: There's always room for improvement
3. **Understanding > Accuracy**: Know why your model makes predictions
4. **Documentation Matters**: Track experiments for reproducibility

### 🎯 Challenge Yourself:
Can you improve the model to achieve MAE < $1? Try:
- Different algorithms (GBT, Linear Regression)
- Feature engineering
- Hyperparameter tuning with CrossValidator
- Removing outliers

### 📧 Share Your Success!
You've completed a real data science project on 46M+ taxi trips! Share your results and what you learned.

**Remember**: Every expert was once a beginner. Keep practicing and stay curious! 🚀

## 📚 Bonus: Quick Reference Guide

### Common Data Science Code Patterns (Spark MLlib)

```python
# Load data from Unity Catalog
df = spark.table('catalog.schema.table')

# Explore data
display(df.limit(5))        # First 5 rows
df.printSchema()            # Data types
display(df.describe())      # Statistics

# Prepare data
feature_cols = [c for c in df.columns if c != 'target']
ml_data = df.select(feature_cols + ['target']).dropna()

# Split data
train_data, test_data = ml_data.randomSplit([0.8, 0.2], seed=42)

# Assemble and scale features
from pyspark.ml.feature import VectorAssembler, StandardScaler
assembler = VectorAssembler(inputCols=feature_cols, outputCol='features_raw', handleInvalid='skip')
scaler = StandardScaler(inputCol='features_raw', outputCol='features', withMean=True, withStd=True)
train_asm = assembler.transform(train_data)
scaler_model = scaler.fit(train_asm)
train_scaled = scaler_model.transform(train_asm)
test_scaled = scaler_model.transform(assembler.transform(test_data))

# Train model
from pyspark.ml.regression import RandomForestRegressor
model = RandomForestRegressor(featuresCol='features', labelCol='target', numTrees=100)
trained_model = model.fit(train_scaled)

# Evaluate
from pyspark.ml.evaluation import RegressionEvaluator
predictions = trained_model.transform(test_scaled)
evaluator = RegressionEvaluator(labelCol='target', predictionCol='prediction', metricName='mae')
mae = evaluator.evaluate(predictions)

# Track with MLflow
import mlflow, mlflow.spark
with mlflow.start_run():
    mlflow.log_metric('mae', mae)
    mlflow.spark.log_model(trained_model, 'model')
```

### Common Mistakes to Avoid:
1. ❌ Using `.toPandas()` on full dataset → ✅ Keep data in Spark, only collect aggregated results
2. ❌ Using test data for training → ✅ Keep test data separate
3. ❌ Ignoring data exploration → ✅ Always explore before modeling
4. ❌ Not tracking experiments → ✅ Use MLflow or similar tools
5. ❌ Overfitting to training data → ✅ Always validate on test set
6. ❌ Forgetting VectorAssembler → ✅ Spark ML needs features in a single vector column

Good luck on your data science journey! 🌟